In [1]:
import matplotlib.dates as mdates
import pandas as pd
import matplotlib.pyplot as plt

# Funzione per calcolare la media, la deviazione standard percentuale e il numero di stazioni per ogni giorno
def calcola_media_devstd_percentuale(df):
    # Raggruppa i dati per 'data', calcola la media, la deviazione standard e il numero di dati
    risultati = df.groupby('data')['rad'].agg(['mean', 'std', 'count']).reset_index()

    # Calcola la deviazione standard percentuale rispetto alla media
    risultati['dev_std_percentuale'] = (risultati['std'] / risultati['mean']) * 100

    # Rimuove eventuali valori nulli generati da calcoli su una singola stazione
    risultati = risultati.dropna()

    # Rinomina la colonna 'count' come 'num_stazioni' per chiarezza
    risultati.rename(columns={'count': 'num_stazioni'}, inplace=True)

    return risultati

# Carica il file CSV aggregato
df = pd.read_csv('ERA_PV_stazioni_clearsky_giornalieri.csv')  # Inserisci il percorso corretto del file

# Verifica e conversione sicura della colonna 'data' in formato datetime
df['data'] = pd.to_datetime(df['data'], errors='coerce')

# Rimuove eventuali righe con data non valida
df = df.dropna(subset=['data'])

# Identifica le stazioni da mantenere (prime occorrenze prima del 2010)
prime_occorrenze = df.groupby('nome')['data'].min()
stazioni_da_includere = prime_occorrenze[prime_occorrenze < '2005-01-01'].index

# Filtra il DataFrame originale per includere solo le stazioni che compaiono prima del 2010
df_filtrato = df[df['nome'].isin(stazioni_da_includere)]

# Esegui il calcolo di media, deviazione standard percentuale e numero di stazioni
risultati_statistici = calcola_media_devstd_percentuale(df_filtrato)

# Calcolo del numeratore della media mobile pesata (dev_std_percentuale * num_stazioni)
risultati_statistici['numeratore_pesato'] = risultati_statistici['dev_std_percentuale'] * risultati_statistici['num_stazioni']

# Calcolo della somma ponderata su una finestra di 20 giorni (somma pesata)
somma_pesata = risultati_statistici['numeratore_pesato'].rolling(window=20, min_periods=1).sum()

# Calcolo della somma dei pesi (num_stazioni) su una finestra di 20 giorni
somma_pesi = risultati_statistici['num_stazioni'].rolling(window=20, min_periods=1).sum()

# Calcolo della media mobile pesata
risultati_statistici['media_mobile_pesata'] = somma_pesata / somma_pesi

# Crea il grafico
plt.figure(figsize=(10, 6))

# Grafico della deviazione standard percentuale
plt.plot(risultati_statistici['data'], risultati_statistici['dev_std_percentuale'],
         marker='o', color='#00ff00', linestyle='None', label='Dev. Std. Percentuale')

# Grafico della media mobile pesata sui 20 giorni
plt.plot(risultati_statistici['data'], risultati_statistici['media_mobile_pesata'],
         color='#2e8b57', linestyle='-', label='Media Mobile Pesata (20 giorni)')

# Imposta il formato delle etichette dell'asse X per mostrare solo gli anni ogni 4 anni
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.YearLocator(4))  # Mostra una tacca ogni 4 anni
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))  # Mostra solo l'anno come etichetta

# Aggiungi etichette e titolo
plt.xlabel('Anno')
plt.ylabel('Deviazione Standard Percentuale')

# Ruota le etichette sull'asse X per leggibilità
plt.xticks(rotation=45)

# Mostra la leggenda
plt.legend()

# Salva il grafico
plt.savefig('deviazione_std_percentuale_con_media_mobile_pesata.png', bbox_inches='tight')

# Mostra il grafico
plt.show()

# Calcolo della media pesata totale
somma_totale_pesata = (risultati_statistici['dev_std_percentuale'] * risultati_statistici['num_stazioni']).sum()
somma_totale_stazioni = risultati_statistici['num_stazioni'].sum()
media_pesata_totale = somma_totale_pesata / somma_totale_stazioni

# Stampa la media pesata totale
print(f"La media pesata totale della deviazione standard percentuale è: {media_pesata_totale:.2f}%")


FileNotFoundError: [Errno 2] No such file or directory: 'ERA_PV_stazioni_clearsky_giornalieri.csv'